In [14]:

from typing import Annotated
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [15]:
from langchain.chat_models import init_chat_model

In [18]:
llm = init_chat_model("google_genai:gemini-3.8-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [19]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='8d749d8e-216c-427f-9a94-9006a29bf8c2'),
 AIMessage(content=[{'type': 'text', 'text': 'Neil Armstrong', 'extras': {'signature': 'Eu8CCuwCAWkUfROhkWj9aaxdLaYW3C5uvutlfEO27Zvc4cZnz0l3NF8ZZRLbJsau9+wo/N+m65KtOuMdMZ3reKZF1FIustYsKSAohBu6RhqyY6hZ5oA1LpjezMbSt4MTEHAgdDGaNfxQSAJnweXwk3vXhj93JpIqa2OsbMxSu/4zXePeL4scpAL5si+nWPHGtjvscGqGh2PUHIqvR0GbsRjHBOnBRHUb8kW7m3UcBvzrQ+UKJ1t+L1VMX29yea8JSBxuj0sGikevIItZi9qzoVQ4rRHAv/FiAKtBN9WPn/CNRf3s4C7PqbgZ2S2wTXWGiwR1vVUSlLiZLMWHPK1Qz/0Fp3taJ2FK8mh3pWAWhAjxnd8rLSxqL5PnhFx+ar4gFnUj3qRptPKByklpgEssrdcPo/nrg21RnyFxLYbRZlG6F03kScKSMnhR512BfjItFQiI7XY0VLx6iLnSawJOL5N58SMN+vmIzYZeHVxHttkmVg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1070a-8940-77a3-9bba-95193b34ba9d-0', tool_calls=[], invalid_t

In [21]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: [{'type': 'text', 'text': 'Modi', 'extras': {'signature': 'EtIDCs8DAWkUfRNhixqiaJbzJ28LAoe7KWjWmgfOKacJ741Vwj8TLuia6mgU/ykfcX7Vu+Quf7lZkD84mbuDxAtiuRGpAFGkyV1lfG04/pIxnjGxKPREZUvXESwXLHevUzT2UGPZU9lYtmC7FeMKQ73Nr7hrsEK1sMASyG8Htyo2zL5+tdvD1dbY4aFCXLI8xtCg+KNHKhFmFcTtKvePi6Uqjug1BCFQmHej1khsxJ9jOVwxVjY+IIs+bCpAepZxWEklDEjS88vt7B9DjTQBO1qy9HrUq/s2QfolQCyfKiaBevuMQ03HBdsjPTvl+MWQ7xwOs9+IJUyd+zkRr0SnD2NElvrL3blWAWjMeDD4tIQJE6IqzuZUcjcUYAF6qGyaGsn2DI3w7j98KD1fNfdHVUBDwTEcSbluDNVF/Bo4JODvecg2V5y4wB6wtWCdMNjkhgUoUkWlYXjFzUMjpUm9wf+l9r4co0NkdNtwxYOGyGYwhdKOgCR+WeWjV4hh8/a0N2qdGabUscDJPQ+YmsV29bMBmFeLDmz9t1l/3/UkQ6pXXOuMXeafjS9mWBgmMmr5Fdws0jByHDp9JG6t+q6kEMaSqMSntAld1xO8HDWeqXdZkBROWA=='}}]
Bot: [{'type': 'text', 'text': '2014', 'extras': {'signature': 'EsEDCr4DAWkUfRPBOiZykcPjwrucsooZH+Ed5q/UBmyzbhQTeQslc19FX2e/QRS8QY1X36ZunGQ/BiaMuLti7LlKG2260nON5yIykYV8yu9oWn1V8ICYxMP0ZTxKQCxf5QIkw2zLKvPigCGxYhhdOgrXJXnItmb2x1fFwj09ahc2UVqOY2qe0Sd9a+NhLvvkPBkIhTuQCkxQnM0zljJYl3F9WgatpmQ60y4lp8GX+dm076A


KeyboardInterrupt

